In [1]:
import simplipy as sp

engine = sp.SimpliPyEngine.from_config("assets/engines/acj-4-3/config.yaml")
engine

In [2]:
engine.simplify("x0 + x0")

'2*x0'

In [3]:
[engine.simplify(e) for e in [
    "x0 * x0 / x0",
    "x0 / (1 / x1)",
    "abs(x0) * abs(x0)",
    "neg(neg(x0))",
    "x0 + 0",
]]

['x0', 'x0*x1', 'x0^2', 'x0', 'x0']

In [4]:
# simplify answers in the form it was given
engine.simplify("2*x0 + x0"), engine.simplify(["+", "*", "2", "x0", "x0"])

('3*x0', ['*', '3', 'x0'])

In [5]:
from simplipy import Mode

{m.name: engine.simplify("atanh(tanh(x0))", mode=m) for m in Mode}

{'f64': 'atanh(tanh(x0))', 'real': 'x0', 'corpus': 'x0'}

In [6]:
{m.name: engine.simplify("sin(np.pi)", mode=m) for m in Mode}

{'f64': 'sin(pi)', 'real': '0', 'corpus': '0'}

In [7]:
# the modes are an axis, not a ladder
try:
    Mode.f64 < Mode.real
except TypeError as e:
    print(e)

'<' not supported between instances of 'Mode' and 'Mode'


In [8]:
expr = "2*x0 + sin(x1)"

engine.to_prefix(expr), engine.to_tagged(expr), engine.to_realization(expr)

(['+', '*', '2', 'x0', 'sin', 'x1'],
 ['<add>', '<mul>', '2', 'x0', '</mul>', 'sin', 'x1', '</add>'],
 ['+', '*', '2', 'x0', 'simplipy.operators.sin', 'x1'])

In [9]:
# every notation reads back to the same expression
engine.to_infix(engine.to_prefix(expr)) == engine.to_infix(engine.to_tagged(expr)) == engine.to_infix(engine.to_realization(expr))

True

In [10]:
# read_infix tolerates vocabulary the engine does not know
engine.read_infix("gamma(x0) + 1")

['+', 'gamma', 'x0', '1']

In [11]:
engine.is_valid("x0 + x1"), engine.is_valid("x0 + ")

(True, False)

In [12]:
[engine.complexity(engine.to_prefix(e)) for e in ["x0", "2*x0", "sin(x0) + cos(x0)"]]

[8000, 19000, 40000]

In [13]:
# numeric folding is opt-in, never part of simplify
engine.simplify("tan(1) + x0"), engine.evaluate_constants("tan(1) + x0")

('x0 + tan(1)', '1.5574077246549023 + x0')

In [14]:
engine.mask("3*x0 + pow(x1, 2)", "fittable"), engine.mask("3*x0 + pow(x1, 2)", "all")

('pow(x1, 2) + <constant> * x0', 'pow(x1, <constant>) + <constant> * x0')

In [15]:
f = engine.as_callable("x0*x1 + 2")

engine.expression_variables("x0*x1 + 2"), f(3.0, 4.0)

(['x0', 'x1'], 14.0)

In [16]:
len(engine.simplification_rules), engine.simplification_rules[0]

(5319, (('sinh', 'asinh', '_0'), ('_0',)))

In [17]:
ops = {
    "+":   {"realization": "+", "alias": [], "arity": 2, "precedence": 1, "commutative": True},
    "*":   {"realization": "*", "alias": [], "arity": 2, "precedence": 2, "commutative": True},
    "neg": {"realization": "simplipy.operators.neg", "alias": [], "arity": 1, "precedence": 2.5, "commutative": False},
    "inv": {"realization": "simplipy.operators.inv", "alias": [], "arity": 1, "precedence": 4, "commutative": False},
}

miner = sp.SimpliPyEngine(operators=ops, rules=[])
miner.find_rules(max_source_pattern_length=3, dummy_variables=1,
                 extra_internal_terms=["0", "1", "<constant>"], X=128, seed=7)
miner.simplification_rules

[(('+', '!0', 'neg', '!0'), ('0',)),
 (('*', '0', '!0'), ('0',)),
 (('*', '$0', 'inv', '$0'), ('1',))]

In [18]:
miner.simplify("x0 + neg(x0)"), miner.simplify("x0 * inv(x0)")

('0', '1')